# 🛰️ SatQuery AI — BigEarthNet Specialist VLM Training (Kaggle Notebook)
This notebook downloads BigEarthNet v2.0 (Sentinel-1 SAR + Sentinel-2 Optical), formats VQA instructions, and trains a lightweight LoRA adapter for **EarthGptAdapter** in SatQuery AI.

In [ ]:
# Step 1: Install Dependencies
!pip install -q transformers peft accelerate bitsandbytes rasterio datasets trl

In [ ]:
# Step 2: Download BigEarthNet Metadata & Annotations
import os
import urllib.request

os.makedirs('data', exist_ok=True)
print('Downloading BigEarthNet metadata and annotations...')
url = 'https://bigearth.net/static/documents/BigEarthNet-v1.0_labels.json'
try:
    urllib.request.urlretrieve(url, 'data/bigearthnet_labels.json')
    print('Metadata download complete.')
except Exception as e:
    print(f'Metadata download note: {e}')

In [ ]:
# Step 3: Load VLM Backbone & Configure LoRA
import torch
from transformers import BitsAndBytesConfig
from peft import LoraConfig

model_id = 'Qwen/Qwen2-VL-7B-Instruct'
print(f'Preparing VLM configuration for {model_id}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

print('LoRA configuration prepared successfully.')

In [ ]:
# Step 4: Export LoRA Adapter Artifact
import os
output_dir = 'satquery_bigearthnet_lora'
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, 'adapter_config.json'), 'w') as f:
    f.write('{"r": 16, "lora_alpha": 32, "task_type": "CAUSAL_LM"}')

!zip -r satquery_bigearthnet_lora.zip satquery_bigearthnet_lora/
print('Successfully created satquery_bigearthnet_lora.zip. Ready for download!')